# 8.7 — Recurrent Neural Networks

An RNN reuses one small transition at every time step, letting a hidden state become the running memory of a sequence.

This is the first neural sequence model in the path: the same computation is applied again and again as tokens arrive. Recurrence relations define the hidden state by combining the previous state with the current input. Parameter sharing lets one transition handle sequences longer than the examples used during training.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 87
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=axis, keepdims=True)


def pad_sequences(sequences, pad_value=0):
    width = max(len(seq) for seq in sequences)
    out = np.full((len(sequences), width), pad_value, dtype=int)
    mask = np.zeros((len(sequences), width), dtype=float)
    for row, seq in enumerate(sequences):
        out[row, : len(seq)] = seq
        mask[row, : len(seq)] = 1.0
    return out, mask


def summarize_rung(rung):
    lengths = [len(seq) for seq in rung["sequences"]]
    vocab = sorted({token for seq in rung["sequences"] for token in seq})
    return {
        "name": rung["name"],
        "n": len(rung["sequences"]),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "vocab": vocab,
    }


def build_sequence_classification_ladder(kind):
    base = [
        {"name": "D1 lesson toy", "sequences": [[1, 0, 1, 1]], "labels": [1], "dependency": 4},
        {"name": "D2 clean patterns", "sequences": [[1, 0, 1], [0, 0, 1], [1, 1, 0], [0, 1, 0], [1, 0, 0]], "labels": [1, 0, 1, 0, 0], "dependency": 3},
        {"name": "D3 long gaps and noise", "sequences": [[1, 0, 0, 1], [1, 2, 0, 0], [0, 2, 2, 1], [0, 0, 0, 0], [1, 0, 2, 1], [0, 1, 2, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 4},
        {"name": "D4 click/dialogue snippets", "sequences": [[3, 1, 0, 4], [2, 0, 4], [3, 2, 2, 4], [1, 1, 0, 0], [3, 0, 1, 4], [2, 2, 0, 4]], "labels": [1, 0, 1, 0, 1, 0], "dependency": 5},
        {"name": "D5 delayed dependency", "sequences": [[1, 0, 0, 0, 0, 4], [0, 0, 1, 0, 0, 4], [1, 2, 2, 2, 2, 0], [0, 2, 2, 2, 2, 4], [1, 0, 2, 0, 2, 4], [0, 0, 0, 0, 0, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 6},
    ]
    if kind == "lstm":
        base[0] = {"name": "D1 lesson gate toy", "sequences": [[2, 0, 2]], "labels": [1], "dependency": 3}
    if kind == "gru":
        base[0] = {"name": "D1 lesson interpolation toy", "sequences": [[2, 1, 0, 1]], "labels": [1], "dependency": 4}
    return base


def build_seq2seq_ladder():
    return [
        {"name": "D1 lesson reverse", "pairs": [([1, 2, 3], [3, 2, 1])], "length": 3},
        {"name": "D2 clean copy/reverse", "pairs": [([1, 2], [2, 1]), ([2, 3], [3, 2]), ([1, 1, 2], [2, 1, 1]), ([3, 1], [1, 3]), ([2, 2, 3], [3, 2, 2])], "length": 3},
        {"name": "D3 unequal reorder", "pairs": [([4, 1, 2], [2, 1]), ([4, 2, 3, 1], [1, 3, 2]), ([1, 4], [1]), ([2, 1, 3], [3, 1, 2]), ([3, 4, 2], [2, 3])], "length": 4},
        {"name": "D4 command pairs", "pairs": [([5, 1, 9], [9, 1]), ([5, 2, 8], [8, 2]), ([6, 1, 7], [7, 1]), ([5, 3, 9], [9, 3]), ([6, 2, 8], [8, 2])], "length": 3},
        {"name": "D5 longer delayed EOS", "pairs": [([1, 0, 0, 2, 9], [2, 1, 10]), ([3, 0, 4, 0, 9], [4, 3, 10]), ([2, 2, 0, 1, 9], [1, 2, 2, 10]), ([5, 0, 0, 0, 6, 9], [6, 5, 10]), ([7, 1, 0, 8, 9], [8, 1, 7, 10])], "length": 6},
    ]


def build_attention_ladder():
    return [
        {"name": "D1 illustrative QKV", "source": [1, 2, 3], "targets": [1], "gold": [1]},
        {"name": "D2 clean alignments", "source": [1, 2, 3, 4], "targets": [1, 2, 3, 4, 2], "gold": [0, 1, 2, 3, 1]},
        {"name": "D3 distractor reorder", "source": [5, 1, 9, 2, 3], "targets": [1, 2, 3], "gold": [1, 3, 4]},
        {"name": "D4 QA translation toy", "source": [7, 4, 8, 4, 9, 2], "targets": [4, 9, 2], "gold": [1, 4, 5]},
        {"name": "D5 diffuse distractors", "source": [8, 0, 4, 0, 9, 0, 2, 0, 4, 1], "targets": [9, 2, 1, 4], "gold": [4, 6, 9, 2]},
    ]


def build_transformer_ladder():
    return [
        {"name": "D1 3-token toy", "sequences": [[1, 2, 1]], "labels": [1], "length": 3},
        {"name": "D2 clean sentence patterns", "sequences": [[1, 3, 2], [2, 3, 1], [1, 4, 4], [2, 4, 4], [1, 3, 3]], "labels": [1, 0, 1, 0, 1], "length": 3},
        {"name": "D3 order conflicts", "sequences": [[5, 1, 9, 2], [5, 2, 9, 1], [1, 0, 2, 0], [2, 0, 1, 0], [1, 9, 9, 2]], "labels": [1, 0, 1, 0, 1], "length": 4},
        {"name": "D4 inline classification corpus", "sequences": [[7, 1, 4, 2, 8], [7, 2, 4, 1, 8], [1, 6, 6, 2, 9], [2, 6, 6, 1, 9], [1, 4, 4, 2, 8]], "labels": [1, 0, 1, 0, 1], "length": 5},
        {"name": "D5 longer sequences", "sequences": [[1, 0, 0, 3, 0, 2], [2, 0, 0, 3, 0, 1], [9, 1, 0, 0, 2, 8], [9, 2, 0, 0, 1, 8], [1, 4, 0, 4, 0, 2]], "labels": [1, 0, 1, 0, 1], "length": 6},
    ]


def accuracy(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

A vanilla RNN reuses the same transition at every time step: $$h_t=\tanh(W_{hh}h_{t-1}+W_{xh}x_t+b_h)$$. Reuse is the point: one small cell can read a length four lesson toy or a longer delayed-dependency sequence.

In [ ]:

def simple_rnn(xs, params):
    h = float(params.get("h0", 0.0))
    states = []
    for x_t in xs:
        preactivation = params["W_hh"] * h + params["W_xh"] * float(x_t) + params.get("b", 0.0)
        h = math.tanh(preactivation)
        states.append(h)
    return np.asarray(states)

lesson_params = {"W_hh": 0.8, "W_xh": 0.6, "b": 0.0, "h0": 0.0}
lesson_states = simple_rnn([1, 0, 1, 1], lesson_params)
expected = np.asarray([0.5370495669980352, 0.40502011794967835, 0.7277917876299974, 0.8281545644800706])
assert np.allclose(lesson_states, expected)
assert np.isclose(lesson_states[-1], 0.8281545644800706)
lesson_states


The same recurrence can feed a sequence classifier by reading the final state $h_T$, or a next-token model by reading intermediate states. The lesson's pitfall is that these states are shifted differently for different tasks.

In [ ]:

def rnn_sequence_classifier(seq, params, threshold=0.55):
    states = simple_rnn(seq, params)
    score = float(states[-1])
    prediction = int(score >= threshold)
    return prediction, score, states

prediction, score, states = rnn_sequence_classifier([1, 0, 1, 1], lesson_params)
assert prediction == 1
print("final h_T", score)


## The dataset ladder

Family F7 uses inline synthetic token sequences. The rungs increase length, vocabulary, noise, and delayed structure while keeping the metric as classification accuracy.

In [ ]:

rungs = build_sequence_classification_ladder("rnn")
for rung in rungs:
    print(summarize_rung(rung))
    print("sample", rung["sequences"][0], "label", rung["labels"][0])


## Run the same method across D1--D5

The RNN cell is unchanged across rungs. Only sequence complexity changes.

In [ ]:

params = {"W_hh": 0.82, "W_xh": 0.55, "b": -0.05, "h0": 0.0}
results = []
hidden_panels = []
for rung in rungs:
    preds = []
    scores = []
    for seq in rung["sequences"]:
        pred, score, states = rnn_sequence_classifier(seq, params, threshold=0.48)
        preds.append(pred)
        scores.append(score)
        hidden_panels.append((rung["name"], states))
    acc = accuracy(preds, rung["labels"])
    results.append({"rung": rung["name"], "dependency": rung["dependency"], "accuracy": acc})

for row in results:
    print(f'{row["rung"]:28s} dependency={row["dependency"]:2d} accuracy={row["accuracy"]:.3f}')


## Results visualization

The closing figure has hidden-state/time heatmaps and the required metric curve.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for index, rung in enumerate(rungs):
    seq = rung["sequences"][0]
    states = simple_rnn(seq, params)
    axes[0, index].imshow(states.reshape(1, -1), aspect="auto", cmap="viridis")
    axes[0, index].set_title(rung["name"].split()[0])
    axes[0, index].set_xlabel("time")
    axes[0, index].set_yticks([])

xs = [row["dependency"] for row in results]
ys = [row["accuracy"] for row in results]
axes[1, 0].plot(xs, ys, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("dependency length")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_title("metric curve")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on D5: memory does not last automatically

The lesson powers $0.2^{20}$ and $0.5^{20}$ show why old evidence can vanish. On the hardest rung we compare a weak recurrent path to a stronger keep path and a gated-cell style shortcut.

In [ ]:

weak_memory = 0.2 ** 20
medium_memory = 0.5 ** 20
strong_memory = 0.9 ** 20
assert np.isclose(weak_memory, 1.0485760000000012e-14)
assert np.isclose(medium_memory, 9.5367431640625e-07)
assert np.isclose(strong_memory, 0.12157665459056935)

bad_params = {"W_hh": 0.2, "W_xh": 0.55, "b": -0.05, "h0": 0.0}
good_params = {"W_hh": 0.92, "W_xh": 0.55, "b": -0.05, "h0": 0.0}
d5 = rungs[-1]
bad_preds = [rnn_sequence_classifier(seq, bad_params, threshold=0.48)[0] for seq in d5["sequences"]]
good_preds = [rnn_sequence_classifier(seq, good_params, threshold=0.48)[0] for seq in d5["sequences"]]
print("wrong low-memory D5 accuracy", accuracy(bad_preds, d5["labels"]))
print("fixed stronger keep-path D5 accuracy", accuracy(good_preds, d5["labels"]))


## Evaluate it + Practice

- Metric: sequence classification accuracy.
- No-skill baseline: majority class on each rung.
- Cheap sanity check: D1 final state asserts to 0.8281545644800706.
- Ablation: set W_hh small and watch delayed accuracy drop.
- Failure signals: hidden heatmaps saturate or decay to zero.

Practice 1: Change one rung so the dependency is one step farther away and predict how the metric curve should move.

Practice 2: Convert the classifier into a next-token model and align predictions with shifted targets.

Practice 3: Add one distractor token and decide whether it should change $h_T$.